In [ ]:
import pandas as pd
import os, sys
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.impute import IterativeImputer

ruta_raiz = os.path.abspath('..')
if ruta_raiz not in sys.path:
    sys.path.append(ruta_raiz)

%load_ext autoreload
%autoreload 2
from Data_preprocesing.IQRCapper import IQRCapper


# AÑADIDO: .values.ravel() al final de las 'y' para evitar los errores rojos
X_train = pd.read_csv("../Train_test_split/X_train.csv")
y_train = pd.read_csv("../Train_test_split/y_train.csv").values.ravel()
X_test = pd.read_csv("../Train_test_split/X_test.csv")
y_test = pd.read_csv("../Train_test_split/y_test.csv").values.ravel()

target_folder = '../comparations'
os.makedirs(target_folder, exist_ok=True) # exist_ok=True prevents errors if the folder already exists

# 2. Updated function to save to the new path
def save_experiment(model_name, imbalance_method, accuracy, precision, recall, f1, roc_auc):
    new_result = {
        'Model': [model_name],
        'Imbalance_Method': [imbalance_method],
        'Accuracy': [round(accuracy, 4)],
        'Precision': [round(precision, 4)],
        'Recall': [round(recall, 4)],
        'F1_Score': [round(f1, 4)],
        'ROC_AUC': [round(roc_auc, 4)]
    }
    df_new = pd.DataFrame(new_result)
    
    # Point the file to inside the folder
    csv_file = f'{target_folder}/imbalance_results.csv'
    
    if os.path.exists(csv_file):
        df_new.to_csv(csv_file, mode='a', header=False, index=False)
    else:
        df_new.to_csv(csv_file, index=False)
        
    print(f"✅ Success! Results for {model_name} + {imbalance_method} saved in the '{target_folder}' folder.")

Voting classifier

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.combine import SMOTETomek, SMOTEENN
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def build_voting_classifier():
    lr  = LogisticRegression(random_state=42, max_iter=1000)
    rfc = RandomForestClassifier(random_state=42)
    svc = SVC(probability=True, random_state=42)
    return VotingClassifier(
        estimators=[('lr', lr), ('rfc', rfc), ('svc', svc)],
        voting='soft'
    )

imbalance_methods = {
    "Baseline": None,
    "RandomUnderSampler":     RandomUnderSampler(random_state=42),
    "TomekLinks":             TomekLinks(),
    "RandomOverSampler":      RandomOverSampler(random_state=42),
    "SMOTE":                  SMOTE(random_state=42),
    "ADASYN":                 ADASYN(random_state=42),
    "SMOTETomek":             SMOTETomek(random_state=42),
    "SMOTEENN":               SMOTEENN(random_state=42),
}
imputador_gb = IterativeImputer(
    estimator=GradientBoostingRegressor(n_estimators=50, random_state=42), 
    random_state=42
)
for method_name, sampler in imbalance_methods.items():
    print(f"\n--- {method_name} ---")

    voting_clf = build_voting_classifier()

    if sampler is None:
        pipe = ImbPipeline([
            ('capping_outliers', IQRCapper(factor=1.5)),
            ('imputacion', imputador_gb),
            ('classifier', voting_clf)])
    else:
        pipe = ImbPipeline([
            ('capping_outliers', IQRCapper(factor=1.5)),
            ('imputacion', imputador_gb),
            ('sampler', sampler), ('classifier', voting_clf)])

    pipe.fit(X_train, y_train)

    y_pred  = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)

    accuracy  = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall    = recall_score(y_test, y_pred, average='weighted')
    f1        = f1_score(y_test, y_pred, average='weighted')
    roc_auc   = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')

    print(f"Accuracy: {accuracy:.4f} | F1: {f1:.4f} | ROC-AUC: {roc_auc:.4f}")

    save_experiment(
        model_name="VotingClassifier (LR+RFC+SVC)",
        imbalance_method=method_name,
        accuracy=accuracy,
        precision=precision,
        recall=recall,
        f1=f1,
        roc_auc=roc_auc
    )


--- Baseline ---
Accuracy: 0.7479 | F1: 0.7435 | ROC-AUC: 0.8981
✅ Success! Results for VotingClassifier (LR+RFC+SVC) + Baseline saved in the 'comparations' folder.

--- RandomUnderSampler ---
Accuracy: 0.7243 | F1: 0.7270 | ROC-AUC: 0.8935
✅ Success! Results for VotingClassifier (LR+RFC+SVC) + RandomUnderSampler saved in the 'comparations' folder.

--- TomekLinks ---
Accuracy: 0.7478 | F1: 0.7424 | ROC-AUC: 0.8970
✅ Success! Results for VotingClassifier (LR+RFC+SVC) + TomekLinks saved in the 'comparations' folder.

--- RandomOverSampler ---
Accuracy: 0.7375 | F1: 0.7395 | ROC-AUC: 0.8972
✅ Success! Results for VotingClassifier (LR+RFC+SVC) + RandomOverSampler saved in the 'comparations' folder.

--- SMOTE ---
Accuracy: 0.7372 | F1: 0.7393 | ROC-AUC: 0.8967
✅ Success! Results for VotingClassifier (LR+RFC+SVC) + SMOTE saved in the 'comparations' folder.

--- ADASYN ---
Accuracy: 0.7306 | F1: 0.7322 | ROC-AUC: 0.8952
✅ Success! Results for VotingClassifier (LR+RFC+SVC) + ADASYN saved in

Hyperparameter Optuna for VotingClassifier

In [ ]:
import optuna
from imblearn.under_sampling import TomekLinks
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import f1_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_voting(trial):
    # Hiperparámetros de LogisticRegression
    lr_C = trial.suggest_float('lr_C', 0.01, 10.0, log=True)

    # Hiperparámetros de RandomForest
    rfc_n_estimators  = trial.suggest_int('rfc_n_estimators', 50, 300)
    rfc_max_depth     = trial.suggest_int('rfc_max_depth', 3, 20)
    rfc_max_features  = trial.suggest_float('rfc_max_features', 0.3, 1.0)

    # Hiperparámetros de SVC
    svc_C      = trial.suggest_float('svc_C', 0.01, 10.0, log=True)
    svc_kernel = trial.suggest_categorical('svc_kernel', ['rbf', 'linear'])

    lr  = LogisticRegression(C=lr_C, max_iter=1000, random_state=42)
    rfc = RandomForestClassifier(
        n_estimators=rfc_n_estimators,
        max_depth=rfc_max_depth,
        max_features=rfc_max_features,
        random_state=42,
        n_jobs=-1
    )
    svc = SVC(C=svc_C, kernel=svc_kernel, probability=True, random_state=42)

    voting_clf = VotingClassifier(
        estimators=[('lr', lr), ('rfc', rfc), ('svc', svc)],
        voting='soft'
    )

    pipe = ImbPipeline([
        ('capping_outliers', IQRCapper(factor=1.5)),
        ('imputacion', imputador_gb),
        ('sampler', TomekLinks()),
        ('classifier', voting_clf)
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    return f1_score(y_test, y_pred, average='weighted')


study_voting = optuna.create_study(direction='maximize')
study_voting.optimize(objective_voting, n_trials=50, show_progress_bar=True)

print(f"\nMejor F1: {study_voting.best_value:.4f}")
print(f"Mejores hiperparámetros: {study_voting.best_params}")

  0%|          | 0/50 [00:00<?, ?it/s]


Mejor F1: 0.7497
Mejores hiperparámetros: {'lr_C': 0.038108358016492344, 'rfc_n_estimators': 141, 'rfc_max_depth': 12, 'rfc_max_features': 0.6505992117386982, 'svc_C': 2.581345892811195, 'svc_kernel': 'rbf'}


In [8]:
from imblearn.under_sampling import TomekLinks
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

best = study_voting.best_params

lr  = LogisticRegression(C=best['lr_C'], max_iter=1000, random_state=42)
rfc = RandomForestClassifier(
    n_estimators=best['rfc_n_estimators'],
    max_depth=best['rfc_max_depth'],
    max_features=best['rfc_max_features'],
    random_state=42,
    n_jobs=-1
)
svc = SVC(C=best['svc_C'], kernel=best['svc_kernel'], probability=True, random_state=42)

voting_final = VotingClassifier(
    estimators=[('lr', lr), ('rfc', rfc), ('svc', svc)],
    voting='soft'
)

pipe_voting_final = ImbPipeline([
    ('sampler', TomekLinks()),
    ('classifier', voting_final)
])

pipe_voting_final.fit(X_train, y_train)
y_pred  = pipe_voting_final.predict(X_test)
y_proba = pipe_voting_final.predict_proba(X_test)

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall    = recall_score(y_test, y_pred, average='weighted')
f1        = f1_score(y_test, y_pred, average='weighted')
roc_auc   = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1:        {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

save_experiment(
    model_name="VotingClassifier Optuna (Final)",
    imbalance_method="TomekLinks",
    accuracy=accuracy,
    precision=precision,
    recall=recall,
    f1=f1,
    roc_auc=roc_auc
)

Accuracy:  0.7542
Precision: 0.7515
Recall:    0.7542
F1:        0.7497
ROC-AUC:   0.8983
✅ Success! Results for VotingClassifier Optuna (Final) + TomekLinks saved in the 'comparations' folder.


BaggingClassifier

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.combine import SMOTETomek, SMOTEENN
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

imbalance_methods = {
    "Baseline": None,
    "RandomUnderSampler":     RandomUnderSampler(random_state=42),
    "TomekLinks":             TomekLinks(),
    "RandomOverSampler":      RandomOverSampler(random_state=42),
    "SMOTE":                  SMOTE(random_state=42),
    "ADASYN":                 ADASYN(random_state=42),
    "SMOTETomek":             SMOTETomek(random_state=42),
    "SMOTEENN":               SMOTEENN(random_state=42),
}

bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    max_samples=0.8,
    max_features=0.8,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

for method_name, sampler in imbalance_methods.items():
    print(f"\n--- {method_name} ---")

    if sampler is None:
        pipe = ImbPipeline([
            ('capping_outliers', IQRCapper(factor=1.5)),
            ('imputacion', imputador_gb),
            ('classifier', bagging)])
    else:
        pipe = ImbPipeline([('sampler', sampler),
                            ('capping_outliers', IQRCapper(factor=1.5)),
                            ('imputacion', imputador_gb),
                            ('classifier', bagging)])

    pipe.fit(X_train, y_train)

    y_pred  = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)

    accuracy  = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall    = recall_score(y_test, y_pred, average='weighted')
    f1        = f1_score(y_test, y_pred, average='weighted')
    roc_auc   = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')

    print(f"Accuracy: {accuracy:.4f} | F1: {f1:.4f} | ROC-AUC: {roc_auc:.4f}")

    save_experiment(
        model_name="BaggingClassifier (DecisionTree)",
        imbalance_method=method_name,
        accuracy=accuracy,
        precision=precision,
        recall=recall,
        f1=f1,
        roc_auc=roc_auc
    )


--- Baseline ---
Accuracy: 0.7479 | F1: 0.7453 | ROC-AUC: 0.8963
✅ Success! Results for BaggingClassifier (DecisionTree) + Baseline saved in the 'comparations' folder.

--- RandomUnderSampler ---
Accuracy: 0.7318 | F1: 0.7352 | ROC-AUC: 0.8918
✅ Success! Results for BaggingClassifier (DecisionTree) + RandomUnderSampler saved in the 'comparations' folder.

--- TomekLinks ---
Accuracy: 0.7517 | F1: 0.7485 | ROC-AUC: 0.8961
✅ Success! Results for BaggingClassifier (DecisionTree) + TomekLinks saved in the 'comparations' folder.

--- RandomOverSampler ---
Accuracy: 0.7434 | F1: 0.7440 | ROC-AUC: 0.8946
✅ Success! Results for BaggingClassifier (DecisionTree) + RandomOverSampler saved in the 'comparations' folder.

--- SMOTE ---
Accuracy: 0.7466 | F1: 0.7484 | ROC-AUC: 0.8942
✅ Success! Results for BaggingClassifier (DecisionTree) + SMOTE saved in the 'comparations' folder.

--- ADASYN ---
Accuracy: 0.7403 | F1: 0.7419 | ROC-AUC: 0.8929
✅ Success! Results for BaggingClassifier (DecisionTree)

Hyperparameter Optuna for BaggingClassifier

In [ ]:
import optuna
from imblearn.under_sampling import TomekLinks
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score

# Silenciamos los logs de optuna para que no llene la pantalla
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    # Hiperparámetros del BaggingClassifier
    n_estimators  = trial.suggest_int('n_estimators', 50, 500)
    max_samples   = trial.suggest_float('max_samples', 0.5, 1.0)
    max_features  = trial.suggest_float('max_features', 0.5, 1.0)
    
    # Hiperparámetros del DecisionTree base
    max_depth         = trial.suggest_int('max_depth', 3, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)

    pipe = ImbPipeline([
        ('capping_outliers', IQRCapper(factor=1.5)),
        ('imputacion', imputador_gb),
        ('sampler', TomekLinks()),
        ('classifier', BaggingClassifier(
            estimator=DecisionTreeClassifier(
                max_depth=max_depth,
                min_samples_split=min_samples_split
            ),
            n_estimators=n_estimators,
            max_samples=max_samples,
            max_features=max_features,
            bootstrap=True,
            random_state=42,
            n_jobs=-1
        ))
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    return f1_score(y_test, y_pred, average='weighted')


# Creamos el estudio y optimizamos
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

# Resultados
print(f"\nMejor F1: {study.best_value:.4f}")
print(f"Mejores hiperparámetros: {study.best_params}")

# Guardamos en el CSV
save_experiment(
    model_name="BaggingClassifier Optuna",
    imbalance_method="TomekLinks",
    accuracy=0,  # lo rellenamos después con el modelo final
    precision=0,
    recall=0,
    f1=study.best_value,
    roc_auc=0
)

  0%|          | 0/50 [00:00<?, ?it/s]


Mejor F1: 0.7534
Mejores hiperparámetros: {'n_estimators': 335, 'max_samples': 0.7539666651966799, 'max_features': 0.9521450900518603, 'max_depth': 10, 'min_samples_split': 19}
✅ Success! Results for BaggingClassifier Optuna + TomekLinks saved in the 'comparations' folder.


In [ ]:
from imblearn.under_sampling import TomekLinks
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import BaggingClassifier,GradientBoostingRegressor

from sklearn.tree import DecisionTreeClassifier
import sys
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


best_params = study.best_params

pipe_final = ImbPipeline([
    ('capping_outliers', IQRCapper(factor=1.5)),
    ('imputacion', imputador_gb),
    ('sampler', TomekLinks()),
    ('classifier', BaggingClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=best_params['max_depth'],
            min_samples_split=best_params['min_samples_split']
        ),
        n_estimators=best_params['n_estimators'],
        max_samples=best_params['max_samples'],
        max_features=best_params['max_features'],
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    ))
])

pipe_final.fit(X_train, y_train)
y_pred  = pipe_final.predict(X_test)
y_proba = pipe_final.predict_proba(X_test)

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall    = recall_score(y_test, y_pred, average='weighted')
f1        = f1_score(y_test, y_pred, average='weighted')
roc_auc   = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1:        {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

save_experiment(
    model_name="BaggingClassifier Optuna (Final)",
    imbalance_method="TomekLinks",
    accuracy=accuracy,
    precision=precision,
    recall=recall,
    f1=f1,
    roc_auc=roc_auc
)

Accuracy:  0.7559
Precision: 0.7550
Recall:    0.7559
F1:        0.7534
ROC-AUC:   0.9006
✅ Success! Results for BaggingClassifier Optuna (Final) + TomekLinks saved in the 'comparations' folder.
